In [1]:
from pathlib import Path

In [2]:
import pandas as pd 
import numpy as np 

In [3]:
data_folder = Path(".")

attendance_path=data_folder/"attendance.csv"

In [5]:
attendance=pd.read_csv(attendance_path)

In [6]:
attendance.shape

(50000, 5)

In [8]:
attendance.head()

,attendance_id,employee_id,date,status,hours_worked
0,1,1488.0,01/08/2025,Present,8.0
1,2,9422.0,NaN,Leave,15.0
2,3,9777.0,2025/08/02,Present,9.0
3,4,304.0,01/08/2025,NaN,NaN
4,5,892.0,NaN,Present,15.0


In [10]:
attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   attendance_id  50000 non-null  int64  
 1   employee_id    49998 non-null  float64
 2   date           37571 non-null  object 
 3   status         41714 non-null  object 
 4   hours_worked   41672 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.9+ MB


In [11]:
attendance.describe()

,attendance_id,employee_id,hours_worked
count,50000.000000,49998.000000,41672.000000
mean,25000.500000,5007.961998,7.689900
std,14433.901067,3068.435482,5.092028
min,1.000000,1.000000,-1.000000
25%,12500.750000,2489.000000,7.500000
50%,25000.500000,4998.000000,8.000000
75%,37500.250000,7489.000000,9.000000
max,50000.000000,99999.000000,15.000000


In [12]:
attendance.isnull().sum()

attendance_id        0
employee_id          2
date             12429
status            8286
hours_worked      8328
dtype: int64

In [13]:
attendance.duplicated().sum()

0

In [14]:
attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   attendance_id  50000 non-null  int64  
 1   employee_id    49998 non-null  float64
 2   date           37571 non-null  object 
 3   status         41714 non-null  object 
 4   hours_worked   41672 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.9+ MB


In [15]:
(attendance=="").sum()

attendance_id    0
employee_id      0
date             0
status           0
hours_worked     0
dtype: int64

In [17]:
(attendance.apply(lambda x : x.astype(str).str.strip()=="")).sum()

attendance_id    0
employee_id      0
date             0
status           0
hours_worked     0
dtype: int64

In [18]:
clean_attendance=attendance.copy()

In [20]:
clean_attendance["attendance_id"].duplicated().sum()

0

In [21]:
clean_attendance["status"].value_counts(dropna=False)

status
Present    8403
Absent     8366
Leave      8349
present    8310
NaN        8286
WFH        8286
Name: count, dtype: int64

In [22]:
clean_attendance[
(clean_attendance["hours_worked"] < 0 ) | (clean_attendance["hours_worked"]>24)]

,attendance_id,employee_id,date,status,hours_worked
7,8,5205.0,01/08/2025,Absent,-1.0
13,14,2865.0,NaN,WFH,-1.0
26,27,897.0,01/08/2025,Absent,-1.0
30,31,159.0,NaN,NaN,-1.0
36,37,6132.0,NaN,Present,-1.0
...,...,...,...,...,...
49977,49978,1532.0,2025/08/02,Absent,-1.0
49981,49982,9807.0,2025-08-01,Absent,-1.0
49983,49984,1983.0,NaN,Present,-1.0
49984,49985,4264.0,01/08/2025,NaN,-1.0


In [23]:
clean_attendance["date"].value_counts(dropna=False).head(20)

date
2025-08-01    12599
2025/08/02    12561
NaN           12429
01/08/2025    12411
Name: count, dtype: int64

In [24]:
employee_path= data_folder / "employees-cleaned.csv"

In [25]:
employee= pd.read_csv(employee_path)

In [27]:
#For relationship

invalid_employee_ids = clean_attendance[
    ~clean_attendance["employee_id"].isin(employee["employee__id"])
    & clean_attendance["employee_id"].notna()
]
 
invalid_employee_ids

,attendance_id,employee_id,date,status,hours_worked
10841,10842,99999.0,NaN,WFH,9.0
16356,16357,99999.0,2025-08-01,NaN,8.0
17365,17366,99999.0,01/08/2025,NaN,-1.0
27470,27471,99999.0,2025/08/02,Present,8.0
32616,32617,99999.0,NaN,WFH,-1.0
48742,48743,99999.0,NaN,Leave,NaN


In [29]:
clean_attendance = clean_attendance[ clean_attendance["employee_id"].isin(employee["employee__id"])].copy()

In [31]:
clean_attendance[ ~clean_attendance["employee_id"].isin(employee["employee__id"])]

,attendance_id,employee_id,date,status,hours_worked


In [32]:
clean_attendance.columns=(
    clean_attendance.columns
    .str.strip()
    .str.lower()
    .str.replace(r"_+","_",regex=True)
)

In [33]:
clean_attendance.columns

Index(['attendance_id', 'employee_id', 'date', 'status', 'hours_worked'], dtype='object')

In [34]:
clean_attendance=clean_attendance.drop_duplicates().copy()

In [35]:
clean_attendance.duplicated().sum()

0

In [37]:
clean_attendance["employee_id"]=pd.to_numeric(
    clean_attendance["employee_id"],
    errors="coerce"
).astype("Int64")

In [38]:
clean_attendance=clean_attendance[
clean_attendance["employee_id"].isin(employee["employee__id"])].copy()

In [40]:
clean_attendance["date"]=pd.to_datetime(
    clean_attendance["date"],
    errors="coerce",
    dayfirst=True
)

In [41]:
clean_attendance["date"].value_counts(dropna=False).head(20)

date
NaT           37582
2025-08-01    12410
Name: count, dtype: int64

In [42]:
clean_attendance["date"].dtype

dtype('<M8[ns]')

In [43]:
clean_attendance["date"].isna().sum()

37582

In [44]:
clean_attendance["status"]=(
    clean_attendance["status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [45]:
clean_attendance["status"] = clean_attendance["status"].replace({
    "present":"Present",
    "absent":"Absent",
    "wfh":"WFH",
    "leave":"Leave"
})

In [46]:
clean_attendance["status"].value_counts(dropna=False)

status
Present    16711
Absent      8366
Leave       8348
WFH         8284
<NA>        8283
Name: count, dtype: Int64

In [47]:
clean_attendance["status"] = clean_attendance["status"].fillna("Unknown")

In [48]:
clean_attendance["status"].value_counts(dropna=False)

status
Present    16711
Absent      8366
Leave       8348
WFH         8284
Unknown     8283
Name: count, dtype: Int64

In [50]:
clean_attendance["hours_worked"] = pd.to_numeric(
    clean_attendance["hours_worked"],
    errors="coerce"
)

In [54]:
clean_attendance.loc[
    (clean_attendance["hours_worked"] < 0) | (clean_attendance["hours_worked"] > 24), 
    "hours_worked"
] = np.nan

In [56]:
clean_attendance.loc[
clean_attendance["status"].isin(["Absent","Leave"]),"hours_worked"]= clean_attendance.loc[
clean_attendance["status"].isin(["Absent","Leave"]),"hours_worked"].fillna(0)

In [57]:
clean_attendance.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49992 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   attendance_id  49992 non-null  int64         
 1   employee_id    49992 non-null  Int64         
 2   date           12410 non-null  datetime64[ns]
 3   status         49992 non-null  string        
 4   hours_worked   38933 non-null  float64       
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), string(1)
memory usage: 2.3 MB


In [59]:
clean_attendance.isna().sum()

attendance_id        0
employee_id          0
date             37582
status               0
hours_worked     11059
dtype: int64

In [60]:
clean_attendance.duplicated().sum()

0

In [61]:
clean_attendance["status"].value_counts(dropna=False)

status
Present    16711
Absent      8366
Leave       8348
WFH         8284
Unknown     8283
Name: count, dtype: Int64

In [62]:
clean_attendance["hours_worked"].describe()

count    38933.000000
mean         8.442851
std          4.421028
min          0.000000
25%          7.500000
50%          8.000000
75%          9.000000
max         15.000000
Name: hours_worked, dtype: float64

In [65]:
invalid_employee_ids = clean_attendance[

~clean_attendance["employee_id"].isin(employee["employee__id"])]

print("Invalid employeeIds:",len(invalid_employee_ids))

Invalid employeeIds: 0


In [66]:
clean_attendance.to_csv(
    data_folder/"attendance-cleaned.csv",
    index=False
)